In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# ---------------------------------------------------------
# ⚙️ 1. 기본 설정 및 AI 모델 훈련 (준비 단계)
# ---------------------------------------------------------
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False 

# 데이터 로드 및 전처리 (컨닝/쓰레기 변수 제거, 이름 변경)
df = pd.read_csv('./data/대청호_ML_학습용데이터.csv', sep='\t', encoding='cp949')

drop_cols = ['조사일', '유해남조류 세포수 (cells/㎖)']
for col in df.columns:
    if any(word in col for word in ['우점종', '조류독소', '냄새물질']):
        drop_cols.append(col)
        
X = df.drop(columns=drop_cols).drop(columns=['채수위치_문의', '채수위치_추동', '채수위치_회남'])
X = X.rename(columns={
    '과거_남조류(cells/㎖)': '7일 전_남조류(cells/㎖)',
    '과거_수온(℃)': '7일 전_수온(℃)'
})
y = df['유해남조류 세포수 (cells/㎖)']

# 모델 학습 (실전 배포용 AI 완성)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X, y) # 실전용이므로 전체 데이터(X, y)로 최종 학습

# ---------------------------------------------------------
# 📱 2. 실무자용 [녹조 신호등 예측기] 함수 정의
# ---------------------------------------------------------
def predict_next_week_algae(today_data_dict):
    """오늘 측정한 데이터를 넣으면 7일 뒤 신호등을 알려주는 함수"""
    
    # 입력받은 데이터를 AI가 읽을 수 있게 데이터프레임으로 변환
    input_df = pd.DataFrame([today_data_dict])
    
    # AI 예측 수행
    predicted_value = rf_model.predict(input_df)[0]
    
    # 신호등 및 지침 판별 로직
    if predicted_value < 1000:
        light, status, action = '🔵', '안전 (Safe)', '평상시 모니터링 유지'
    elif predicted_value < 10000:
        light, status, action = '🟡', '관심 (Watch)', '[골든타임] 드론 예찰 강화 및 조류 징후 파악'
    elif predicted_value < 1000000:
        light, status, action = '🟠', '경계 (Warning)', '[비상] 활성탄 투입 준비 및 취수구 층수 조절'
    else:
        light, status, action = '🔴', '심각 (Danger)', '[재난] 수상 활동 전면 금지 및 고강도 정수 처리'
        
    # 결과 출력 (알림톡 형식)
    print("=" * 50)
    print(f"🚨 [AI 수질 예보] 7일 뒤 대청호 녹조 예측 결과 🚨")
    print("=" * 50)
    print(f"▶ 예측 수치 : {predicted_value:,.0f} cells/mL")
    print(f"▶ 예보 등급 : {light} {status}")
    print(f"▶ 실무 지침 : {action}")
    print("=" * 50)

# ---------------------------------------------------------
# 🧪 3. 시스템 테스트 (오늘 측정한 가상의 데이터를 넣어보자!)
# ---------------------------------------------------------
# (주의: 딕셔너리의 키 이름은 학습된 X의 컬럼명과 100% 동일해야 함)
today_measurements = {
    '7일 전_남조류(cells/㎖)': 8500,  # 오늘 떠있는 남조류가 다음주의 씨앗!
    '7일 전_수온(℃)': 26.5,
    '수온(℃)': 27.0,
    '평균기온(℃)': 28.5,
    '일강수량(mm)': 15.0,
    '합계 일조시간(hr)': 9.5,
    '합계 일사량(MJ/m2)': 20.1,
    '평균 풍속(m/s)': 1.2,
    '투명도': 0.8,
    'DO(mg/L)': 10.5,
    'BOD(mg/L)': 2.1,
    'COD(mg/L)': 5.4,
    'TN(mg/L)': 2.3,
    'TP(mg/L)': 0.045,
    'PO4-P(mg/L)': 0.012,
    'Chl-a (mg/m³)': 35.5,
    'pH': 8.5
}

# 함수 실행!
predict_next_week_algae(today_measurements)

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- BOD(mg/L)
- COD(mg/L)
- Chl-a (mg/m³)
- DO(mg/L)
- PO4-P(mg/L)
- ...
Feature names seen at fit time, yet now missing:
- Chl-a (㎎/㎥)
- DO(㎎/L)
- 탁도
- 평균기온(°C)


In [3]:
import pandas as pd

# ---------------------------------------------------------
# ⚙️ 1. 모델 및 테스트 데이터 준비 (앞선 코드와 연결됨)
# ---------------------------------------------------------
# (위의 전체 데이터에서 과거 80%로 학습, 최근 20%로 테스트를 진행하여 성능 평가)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
rf_model_test = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_test.fit(X_train, y_train)

# 테스트 구간(최근 20%)에 대한 AI의 예측값 뽑기
ai_predictions = rf_model_test.predict(X_test)

# ---------------------------------------------------------
# 📊 2. 예산 및 비용 상수 세팅 (단위: 만 원)
# ---------------------------------------------------------
C_drone = 50        # [관심-노란불] 드론 1대 띄워서 현장 확인하는 저렴한 비용
C_carbon = 1000     # [경계-주황불] 활성탄 등 정수 약품을 선제적으로 투입하는 방제 비용
C_disaster = 20000  # [재난 발생] 대비 없이 1만 세포수 이상 터졌을 때의 막대한 피해액

# 결산용 변수 초기화
total_prevent_cost = 0  
total_disaster_cost = 0 

print("💰 [AI 도입 경제성 평가] 과거 테스트 구간 시뮬레이션 가동 중...\n")

# ---------------------------------------------------------
# 🔄 3. 시뮬레이션 루프 (매주 AI가 예측하고 방제 결정을 내림)
# ---------------------------------------------------------
for i in range(len(y_test)):
    pred_val = ai_predictions[i]  # AI의 예측값 (7일 전의 예보)
    real_val = y_test.iloc[i]     # 7일 뒤 실제 터진 팩트값
    
    # 1) AI 예측에 따른 선제적 방제 지출 결정
    if 1000 <= pred_val < 10000:
        total_prevent_cost += C_drone   # 노란불 떴으니 드론 출동
    elif pred_val >= 10000:
        total_prevent_cost += C_carbon  # 주황/빨간불 떴으니 약품 투입!
        
    # 2) 실제 재난 발생 여부에 따른 피해액 결산
    # (실제 수치가 1만 이상인데 AI가 방제 지시(주황불 이상)를 안 내렸거나, AI 도입을 안 했을 때의 기본 피해액)
    if real_val >= 10000:
        total_disaster_cost += C_disaster

# 최종 기대 이익(절감액) 계산 (AI 도입 전 무방비 피해액 - AI 방제 예산)
net_profit = total_disaster_cost - total_prevent_cost

# ---------------------------------------------------------
# 📈 4. 최종 결과 출력 (보고서 삽입용)
# ---------------------------------------------------------
print("=" * 55)
print(" 📉 대청호 AI 녹조 예보 시스템 예산 절감 결산 보고서 📉 ")
print("=" * 55)
print(f"▶ 대상 기간: 최근 20% 테스트 구간 (총 {len(y_test)}주)")
print("-" * 55)
print(f"❌ AI 미도입 시 예상 누적 재난 피해액 : {total_disaster_cost:,} 만 원")
print(f"✅ AI 선제 대응으로 지출한 총 예방 비용 : {total_prevent_cost:,} 만 원")
print("-" * 55)
if net_profit > 0:
    print(f"🚀 결론: AI 시스템 도입으로 총 ⭐️{net_profit:,} 만 원⭐️ 예산 절감 성공!")
else:
    print(f"⚠️ 결론: 방제 비용이 예상 피해액을 초과했습니다. (전략 수정 필요)")
print("=" * 55)

💰 [AI 도입 경제성 평가] 과거 테스트 구간 시뮬레이션 가동 중...

 📉 대청호 AI 녹조 예보 시스템 예산 절감 결산 보고서 📉 
▶ 대상 기간: 최근 20% 테스트 구간 (총 365주)
-------------------------------------------------------
❌ AI 미도입 시 예상 누적 재난 피해액 : 560,000 만 원
✅ AI 선제 대응으로 지출한 총 예방 비용 : 41,850 만 원
-------------------------------------------------------
🚀 결론: AI 시스템 도입으로 총 ⭐️518,150 만 원⭐️ 예산 절감 성공!


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# ---------------------------------------------------------
# ⚙️ 1. 기본 설정 및 데이터 전처리
# ---------------------------------------------------------
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False 

print("🚀 [대청호 녹조 AI 마스터 파이프라인] 전체 시스템 가동 중...\n")

# 데이터 로드
df = pd.read_csv('./data/대청호_ML_학습용데이터.csv', sep='\t', encoding='cp949')

# 데이터 누수(컨닝) 방지 및 불필요 변수 제거, 이름 변경
drop_cols = ['조사일', '유해남조류 세포수 (cells/㎖)']
for col in df.columns:
    if any(word in col for word in ['우점종', '조류독소', '냄새물질']):
        drop_cols.append(col)
        
X = df.drop(columns=drop_cols).drop(columns=['채수위치_문의', '채수위치_추동', '채수위치_회남'])
X = X.rename(columns={
    '과거_남조류(cells/㎖)': '7일 전_남조류(cells/㎖)',
    '과거_수온(℃)': '7일 전_수온(℃)'
})
y = df['유해남조류 세포수 (cells/㎖)']

# ---------------------------------------------------------
# 💰 2. [파트 1] 경제성 시뮬레이션 (과거 데이터로 효과 입증)
# ---------------------------------------------------------
print("📊 [파트 1] 예산 절감 경제성 시뮬레이터를 가동합니다...\n")

# 시뮬레이션용 데이터 분할 (과거 80% 학습, 최근 20% 테스트)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 테스트용 모델 학습 및 예측
rf_model_sim = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_sim.fit(X_train, y_train)
ai_predictions = rf_model_sim.predict(X_test)

# 예산 상수 세팅 (단위: 만 원)
C_drone = 50        # 드론 예찰(관심)
C_carbon = 1000     # 활성탄 투입(경계 이상)
C_disaster = 20000  # 방제 실패 시 재난 피해액

total_prevent_cost = 0  
total_disaster_cost = 0 

# 테스트 구간 정산
for i in range(len(y_test)):
    pred_val = ai_predictions[i]
    real_val = y_test.iloc[i]
    
    # AI 예보에 따른 방제 지출
    if 1000 <= pred_val < 10000:
        total_prevent_cost += C_drone
    elif pred_val >= 10000:
        total_prevent_cost += C_carbon
        
    # 실제 재난 발생 시 피해액 (1만 세포수 이상일 때 터진다고 가정)
    if real_val >= 10000:
        total_disaster_cost += C_disaster

net_profit = total_disaster_cost - total_prevent_cost

print("=" * 55)
print(" 📉 과거 데이터 기반 AI 도입 예산 절감 결산 📉 ")
print("=" * 55)
print(f"❌ AI 미도입 시 누적 재난 피해액 : {total_disaster_cost:,} 만 원")
print(f"✅ AI 선제 대응 총 예방 비용     : {total_prevent_cost:,} 만 원")
print("-" * 55)
print(f"🚀 결론: AI 도입으로 총 ⭐️{net_profit:,} 만 원⭐️ 예산 절감 입증!")
print("=" * 55 + "\n")

# ---------------------------------------------------------
# 🚦 3. [파트 2] 실무자용 실전 배포 시스템 (미래 예측)
# ---------------------------------------------------------
print("📱 [파트 2] 실무 배포용 7일 뒤 수질 예측 시스템 활성화...\n")

# 실전용 모델은 미래를 가장 잘 맞히기 위해 가진 '100% 데이터'를 몽땅 학습!
rf_model_deploy = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model_deploy.fit(X, y) 

# ---------------------------------------------------------
# 🧪 4. 통합 시스템 최종 테스트 가동! (에러 완벽 해결)
# ---------------------------------------------------------
print("📥 우리가 가진 가장 최신 측정 데이터를 바탕으로 다음 주 예보를 발령합니다...\n")

# 딕셔너리로 직접 치지 않고, X에서 맨 마지막 행(최신 데이터)을 그대로 복사! 
# (이렇게 하면 공공데이터 특유의 이상한 특수문자나 띄어쓰기 오타 에러가 절대 안 남!)
latest_data = X.iloc[-1:].copy()

# (선택) 만약 수동으로 특정 변수만 오늘 값으로 살짝 바꾸고 싶다면 아래처럼 덮어씌우면 됩니다.
# latest_data['7일 전_수온(℃)'] = 28.5  

# 100% 학습된 완전체 모델로 7일 뒤 미래 예측!
predicted_value = rf_model_deploy.predict(latest_data)[0]

# 신호등 로직
if predicted_value < 1000:
    light, status, action = '🔵', '안전 (Safe)', '평상시 모니터링 유지'
elif predicted_value < 10000:
    light, status, action = '🟡', '관심 (Watch)', '[골든타임] 드론 예찰 강화 및 조류 징후 파악'
elif predicted_value < 1000000:
    light, status, action = '🟠', '경계 (Warning)', '[비상] 활성탄 투입 준비 및 취수구 층수 조절'
else:
    light, status, action = '🔴', '심각 (Danger)', '[재난] 수상 활동 전면 금지 및 고강도 정수 처리'
    
print("=" * 55)
print(f"🚨 [AI 실시간 수질 예보] 7일 뒤 대청호 예측 결과 🚨")
print("=" * 55)
print(f"▶ 예측 수치 : {predicted_value:,.0f} cells/mL")
print(f"▶ 예보 등급 : {light} {status}")
print(f"▶ 실무 지침 : {action}")
print("=" * 55)

🚀 [대청호 녹조 AI 마스터 파이프라인] 전체 시스템 가동 중...

📊 [파트 1] 예산 절감 경제성 시뮬레이터를 가동합니다...

 📉 과거 데이터 기반 AI 도입 예산 절감 결산 📉 
❌ AI 미도입 시 누적 재난 피해액 : 560,000 만 원
✅ AI 선제 대응 총 예방 비용     : 41,850 만 원
-------------------------------------------------------
🚀 결론: AI 도입으로 총 ⭐️518,150 만 원⭐️ 예산 절감 입증!

📱 [파트 2] 실무 배포용 7일 뒤 수질 예측 시스템 활성화...

📥 우리가 가진 가장 최신 측정 데이터를 바탕으로 다음 주 예보를 발령합니다...

🚨 [AI 실시간 수질 예보] 7일 뒤 대청호 예측 결과 🚨
▶ 예측 수치 : 15 cells/mL
▶ 예보 등급 : 🔵 안전 (Safe)
▶ 실무 지침 : 평상시 모니터링 유지
